In [ ]:
import json
import pandas as pd
from pathlib import Path


current_path = Path(__file__).resolve() if '__file__' in locals() else Path('.').resolve()
PROJECT_ROOT = next((p for p in current_path.parents if p.name == "our_project"), current_path)

json_path = PROJECT_ROOT / "evaluations" / "lotte" / "bm25" / "evaluation_report.json"


if not json_path.exists():
    raise FileNotFoundError(f"❌ لم يتم العثور على ملف الـ JSON في المسار الحقيقي: {json_path}")


with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)


print("==================================================================")
print("📌 GLOBAL EVALUATION METRICS SUMMARY - (BM25 Model)")
print("==================================================================")
print(f"📊 Total Evaluated Queries : {data['evaluated_queries']}")
print(f"🚫 Skipped Queries          : {data['skipped_queries']}")
print(f"🎯 Mean Precision@10       : {data['metrics']['mean_precision@10']:.4f}")
print(f"📈 Mean Recall@10          : {data['metrics']['mean_recall@10']:.4f}")
print(f"🏆 Mean Average Precision  : {data['metrics']['map@10']:.4f}  <--- (MAP@10)")
print(f"📉 Mean NDCG@10            : {data['metrics']['mean_ndcg@10']:.4f}")
print("==================================================================\n")


queries_list = []
for q in data["per_query_results"]:
    queries_list.append({
        "Query ID": q["query_id"],
        "Query Text": q["query_text"],
        "Precision@10": q["metrics"]["precision@10"],
        "Recall@10": q["metrics"]["recall@10"],
        "AP@10": q["metrics"]["ap@10"],
        "NDCG@10": q["metrics"]["ndcg@10"]
    })

df_bm25 = pd.DataFrame(queries_list)

print("📋 DETAILED PER-QUERY METRICS (BM25):")
pd.set_option('display.max_rows', None)
display(df_bm25)

📌 GLOBAL EVALUATION METRICS SUMMARY - (BM25 Model)
📊 Total Evaluated Queries : 563
🚫 Skipped Queries          : 0
🎯 Mean Precision@10       : 0.1126
📈 Mean Recall@10          : 0.4343
🏆 Mean Average Precision  : 0.4227  <--- (MAP@10)
📉 Mean NDCG@10            : 0.3708

📋 DETAILED PER-QUERY METRICS (BM25):


,Query ID,Query Text,Precision@10,Recall@10,AP@10,NDCG@10
0,0,do bards have to sing?,0.0,0.000000,0.000000,0.000000
1,1,do spells count as ranged attacks?,0.1,0.333333,0.111111,0.141267
2,2,how much xp do you need to get to level 2 d&d?,0.0,0.000000,0.000000,0.000000
3,3,do tiefling horns grow back?,0.3,1.000000,1.000000,1.000000
4,4,do paladins have to be religious?,0.0,0.000000,0.000000,0.000000
5,5,are natural attacks unarmed strikes?,0.2,1.000000,0.225000,0.441307
6,6,do paladins have to be good?,0.1,0.250000,0.166667,0.139056
7,7,do halflings have pointy ears?,0.2,0.666667,1.000000,0.765361
8,8,do racial bonuses stack pathfinder?,0.1,0.500000,0.100000,0.177239
9,9,do attacks of opportunity stop movement?,0.0,0.000000,0.000000,0.000000


In [6]:
from services.dataset.dataset_loader import (

    load_lotte,

)



dataset = load_lotte()



for index, document in enumerate(

    dataset.docs_iter()

):

    print("=" * 80)

    print("DOC ID:", document.doc_id)

    print("TEXT:")

    print(document.text)



    if index == 4:

      break  

DOC ID: 0
TEXT:
Multiclassing no longer takes an XP hit, and your favored class gives you one of two bonuses at every level: +1 hp +1 skill point Advanced Players Guide add other options for specific Race/Class combos
DOC ID: 1
TEXT:
There's a fairly large smattering of stuff. Here's a taste: Races have all been powered up; +2 to two stats, -2 to one stat (including half-orcs!), with some different abilities for some. In 3.5 terms, they're roughly an ECL of +1 (as opposed to 0 in 3e). No one has a favored class decided by their race now - you choose your favored class, typically the one you level at 1st level, but you don't get an XP penalty for multiclassing too high above that one - you just gain an extra skill point or hit point for leveling your favored class. Many of the classes have had options added, often to fill out "dead levels." Barbarians gain extra "rage powers," sorcerers have bloodlines that give them granted powers, wizards have varying effects for specializing, fighter

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd

# 1. إعداد المسارات لضمان عمل الـ Imports بدون مشاكل
current_path = Path('.').resolve()
PROJECT_ROOT = next((p for p in current_path.parents if p.name == "our_project"), current_path)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# استيراد التوابع
from services.dataset.dataset_loader import load_lotte
try:
    from services.data_base.database_service import get_raw_documents_by_ids
except ImportError:
    from services.data_base import get_raw_documents_by_ids

# ==========================================================
# 2. حددي هنا الـ ID الذي تريدين مطابقته واختباره
# ==========================================================
target_doc_id = "69691" 

print("================================================================================")
print(f"🔍 LIVE VERIFICATION & CROSS-MATCH FOR DOC_ID: {target_doc_id}")
print("================================================================================")

raw_dataset_text = None

# 🏢 [القسم الأول]: جلب النص من الداتا سيت الخام عبر المسار المباشر للملف بترميز UTF-8
try:
    dataset = load_lotte()
    
    # الحصول على المسار الحقيقي لملف الـ collection.tsv المخزن على جهازكِ
    # مكتبة ir_datasets تقوم بتحميل الملف في هذا المسار التلقائي
    source_path = dataset.docs_path()
    
    # قراءة الملف بأمان تام باستخدام pandas مع تحديد ترميز utf-8 صراحة
    # الملف يتكون من عمودين: الـ ID والنص، ويفصل بينهما Tab (\t)
    df_source = pd.read_csv(source_path, sep="\t", names=["doc_id", "text"], dtype={"doc_id": str}, encoding="utf-8")
    
    # البحث عن الـ ID المطلوب وجلب نصه
    match_row = df_source[df_source["doc_id"] == str(target_doc_id)]
    if not match_row.empty:
        raw_dataset_text = match_row["text"].values[0]
        print(f"🟢 [FROM RAW DATASET]:\n{raw_dataset_text}\n")
    else:
        print(f"❌ لم يتم العثور على الـ ID: {target_doc_id} في ملف الداتا سيت الأصلي.\n")
        
except Exception as e:
    print(f"❌ خطأ أثناء الجلب من الداتا سيت الخام: {e}\n")

print("-" * 80)

# 💾 [القسم الثاني]: جلب النص من قاعدة بيانات الـ SQLite الخاصة بمشروعكم
sqlite_text = None
try:
    db_result = get_raw_documents_by_ids([target_doc_id])
    if db_result:
        sqlite_text = db_result[0]['text']
        print(f"🔵 [FROM YOUR SQLITE DB]:\n{sqlite_text}\n")
    else:
        print("❌ لم يتم العثور على المستند داخل الـ SQLite.\n")
except Exception as e:
    print(f"❌ خطأ أثناء الجلب من الـ SQLite: {e}\n")

# 🏆 [القسم الثالث]: النتيجة والـ Verification التلقائي
print("================================================================================")
if raw_dataset_text and sqlite_text:
    if str(raw_dataset_text).strip() == str(sqlite_text).strip():
        print("✅ MATCH RESULT: 100% IDENTICAL! (تطابق تام وبدون أي أخطاء)")
    else:
        print("⚠️ MATCH RESULT: MISMATCH! (هناك اختلاف بين النصوص)")
else:
    print("⚠️ تعذر إكمال المطابقة التلقائية بسبب فشل جلب أحد النصوص.")
print("================================================================================")

🔍 LIVE VERIFICATION & CROSS-MATCH FOR DOC_ID: 69691
🟢 [FROM RAW DATASET]:
Spell attacks A "ranged attack" can also be a spell attack (like fire bolt), while a "ranged weapon attack" can not.

--------------------------------------------------------------------------------
🔵 [FROM YOUR SQLITE DB]:
Spell attacks A "ranged attack" can also be a spell attack (like fire bolt), while a "ranged weapon attack" can not.

✅ MATCH RESULT: 100% IDENTICAL! (تطابق تام وبدون أي أخطاء)
